# Treinar tudo no Colab — dengue

Este é o único notebook que você precisa abrir para executar baselines, preparar as dobras e treinar/selecionar os modelos M1. Use **Ambiente de execução → Executar tudo**. O notebook clona automaticamente o branch publicado `feat/ricardo`, instala o projeto e executa as etapas em sequência.

Na primeira execução, autorize o acesso ao Google Drive. Os dados de 2022 e 2023 serão reutilizados se já estiverem em `MyDrive/dengue-comorbidity-risk/data`; se estiverem faltando, o notebook baixa e converte esses anos automaticamente. Os resultados, o progresso retomável e os artefatos ficam no Drive.

**Não carrega nem treina com 2024.** O conjunto de teste permanece congelado. O treinamento pode demorar; se a sessão do Colab cair, execute o notebook novamente para retomar os modelos concluídos. Não compartilhe OOF, `mlruns` ou arquivos por caso.

In [ ]:
import gc
import json
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        f'Este projeto exige Python 3.13; o Colab está usando '
        f'{sys.version_info.major}.{sys.version_info.minor}. '
        'No Colab, selecione um runtime Python 3.13 e execute novamente.'
    )

from google.colab import drive

drive.mount('/content/drive')

REPOSITORY = 'https://github.com/PabloThePmX/dengue-comorbidity-risk.git'
BRANCH = 'feat/ricardo'
remote = subprocess.run(
    ['git', 'ls-remote', '--heads', REPOSITORY, f'refs/heads/{BRANCH}'],
    check=True,
    capture_output=True,
    text=True,
)
if not remote.stdout.strip():
    raise RuntimeError(
        f'O branch {BRANCH!r} não está acessível no GitHub. '
        'Confirme que o repositório está público e tente novamente.'
    )
commit = remote.stdout.split()[0]
repo = Path(f'/content/dengue-comorbidity-risk-{commit[:10]}')
if not repo.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY, str(repo)],
        check=True,
    )
else:
    local_commit = subprocess.run(
        ['git', '-C', str(repo), 'rev-parse', 'HEAD'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if local_commit != commit:
        raise RuntimeError(
            f'O clone local está no commit {local_commit}, mas o branch remoto mudou '
            f'para {commit}. Reinicie o ambiente do Colab para baixar a versão nova.'
        )

required_files = [
    repo / 'pyproject.toml',
    repo / 'config' / 'base.yaml',
    repo / 'config' / 'esperado.yaml',
    repo / 'src' / 'dengue_sinan' / 'download.py',
    repo / 'src' / 'dengue_sinan' / 'baselines.py',
    repo / 'src' / 'dengue_sinan' / 'folds.py',
    repo / 'src' / 'dengue_sinan' / 'runner.py',
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('Arquivos ausentes no branch clonado: ' + ', '.join(missing_files))

print(f'Instalando o projeto do branch {BRANCH}, commit {commit[:10]}...', flush=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(repo)],
    check=True,
)
os.chdir(repo)

from dengue_sinan.baselines import run_and_log_baselines
from dengue_sinan.config import load_config
from dengue_sinan.dataset import load_population
from dengue_sinan.download import fetch_year, sha256_file
from dengue_sinan.folds import verify_and_save_folds
from dengue_sinan.runner import run_all_models

project_dir = Path('/content/drive/MyDrive/dengue-comorbidity-risk')
data_dir = project_dir / 'data'
reports_dir = project_dir / 'reports'
mlruns_dir = project_dir / 'mlruns'
models_dir = project_dir / 'models'
m1_dir = project_dir / 'm1'
for directory in (data_dir, reports_dir, mlruns_dir, models_dir, m1_dir):
    directory.mkdir(parents=True, exist_ok=True)

config = load_config(repo / 'config' / 'base.yaml')
config['caminhos'].update(
    {
        'dados': str(data_dir),
        'relatorios': str(reports_dir),
        'mlruns': str(mlruns_dir),
        'modelos': str(models_dir),
        'm1': str(m1_dir),
        'dobras': str(m1_dir / 'dobras.npz'),
        'esperado': str(repo / 'config' / 'esperado.yaml'),
    }
)
anos_desenvolvimento = [2022, 2023]
if config['anos_dev'] != anos_desenvolvimento:
    raise ValueError(f'Configuração inesperada de anos de treino: {config["anos_dev"]}')

for ano in anos_desenvolvimento:
    print(f'Verificando/obtendo dados de {ano}...', flush=True)
    entrada = fetch_year(ano, data_dir)
    print(
        f'{ano}: {entrada["linhas"]:,} linhas no arquivo convertido; '
        f'DataSUS {entrada["status_datasus"]}.',
        flush=True,
    )

print('Carregando apenas os anos de desenvolvimento 2022 e 2023...', flush=True)
df_dev = load_population(config, anos=anos_desenvolvimento)
anos_carregados = sorted(df_dev['ANO'].dropna().astype(int).unique().tolist())
if anos_carregados != anos_desenvolvimento or 2024 in set(df_dev['ANO'].astype(int)):
    raise RuntimeError(f'Anos carregados não permitidos para treino: {anos_carregados}')

expected = load_config(repo / 'config' / 'esperado.yaml')
fold_path, fold_hash, fold_checks = verify_and_save_folds(
    df_dev,
    m1_dir / 'dobras.npz',
    reports_dir,
    expected['eventos_por_regiao'],
    config['ordem_dobras'],
)
if not all(check['passou'] for check in fold_checks):
    raise RuntimeError(f'A validação das dobras falhou: {fold_checks}')
print(f'Dobras verificadas; hash {fold_hash}.', flush=True)

manifest = json.loads((data_dir / 'manifest.json').read_text(encoding='utf-8'))
baseline_marker = m1_dir / 'baselines-2022-2023.json'
baseline_files = [mlruns_dir / 'oof' / f'{name}.parquet' for name in ('b0', 'b1', 'b2')]
baseline_state = {
    'commit': commit,
    'hashes_dados': {
        str(ano): sha256_file(data_dir / manifest[str(ano)]['parquet'])
        for ano in anos_desenvolvimento
    },
}
baseline_reusable = (
    baseline_marker.is_file()
    and all(path.is_file() for path in baseline_files)
    and json.loads(baseline_marker.read_text(encoding='utf-8')) == baseline_state
)
if baseline_reusable:
    print('Baselines válidos já existem no Drive; vou reutilizá-los.', flush=True)
else:
    print('Executando baselines B0, B1 e B2...', flush=True)
    run_and_log_baselines(df_dev, config)
    temporary_marker = baseline_marker.with_name(baseline_marker.name + '.tmp')
    temporary_marker.write_text(
        json.dumps(baseline_state, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    temporary_marker.replace(baseline_marker)
    print('Baselines concluídos e salvos no Drive.', flush=True)

del df_dev
gc.collect()
print('Iniciando treinamento/seleção M1. O progresso é salvo no Drive para retomada...', flush=True)
resultado = run_all_models(config)

session_report_path = reports_dir / 'verificacao' / 'sessao-m1.json'
session_report = json.loads(session_report_path.read_text(encoding='utf-8'))
if session_report.get('passou') is not True or session_report.get('sessao_completa') is not True:
    raise RuntimeError(
        f'A sessão terminou sem passar todas as verificações. Consulte {session_report_path}'
    )

print('\nTREINAMENTO CONCLUÍDO')
print(f'Modelo selecionado: {resultado["modelo"]}')
print(f'Bundle final: {resultado["artefato"]}')
print(f'Relatório de sessão: {session_report_path}')
print(f'Relatório de baselines: {reports_dir / "baselines.md"}')
print('Alias MLflow: models:/dengue-m1@melhor-validacao')
print('O conjunto de teste de 2024 não foi carregado.')